# Tariikhna — Generate Narration Audio (edge-tts)

Generates one MP3 per panel from each panel's `narrative_text`, using Microsoft
Edge neural voices — **free, no API key, no character limits**.

Routes files into two folders (same pattern as the image notebook):
```
output_base/   <- files WITHOUT _1
output_v1/     <- files WITH _1
```
Inside each: `audio/` (the MP3s) and `stories/` (JSONs with an `audio_file` field).

Works in **Colab** or **local Jupyter / VS Code notebooks**. Needs internet
(calls Microsoft's free endpoint) but no account.

## 1. Install

In [ ]:
!pip install edge-tts -q
import os, json, glob, asyncio
import edge_tts
print('edge-tts ready')

## 2. Configuration

In [ ]:
# ---- Voice choice ----
# A warm storyteller voice suits a children's comic. Good options:
#   en-US-JennyNeural   gentle, pleasant female - great for storytelling (recommended)
#   en-US-AriaNeural    warm, friendly female - general narrator
#   en-US-AnaNeural     CHILD voice (young girl) - cute, very kid-friendly
#   en-US-DavisNeural   calm, soothing male - bedtime-story feel
#   en-GB-SoniaNeural   British female, warm storybook feel
VOICE = 'en-GB-SoniaNeural'   # British female, warm storybook feel

# Slightly slower pacing helps young listeners. '-10%' = 10% slower.
RATE = '-10%'
PITCH = '+0Hz'   # '+0Hz' = unchanged

# Output folders
BASE_OUT = 'output_base'
V1_OUT = 'output_v1'

print(f'Voice: {VOICE}  (rate {RATE})')

## 3. (Optional) Browse Available Voices

Run this to see all English voices and pick a different one.

In [ ]:
async def list_english_voices():
    voices = await edge_tts.list_voices()
    for v in sorted(voices, key=lambda x: x['ShortName']):
        if v['ShortName'].startswith('en-'):
            tags = ', '.join(v.get('VoiceTag', {}).get('VoicePersonalities', []))
            print(f"  {v['ShortName']:28s} {v['Gender']:8s} {tags}")

# Uncomment to run:
# await list_english_voices()    # in Jupyter/Colab you can await at top level
# If 'await' errors, use:  asyncio.run(list_english_voices())

## 4. Upload Your Story JSONs

**Colab:** run the upload cell.  
**Local Jupyter:** skip the upload, just put your JSONs in an `input_stories/` folder
next to this notebook (and set INPUT_DIR below).

In [ ]:
INPUT_DIR = 'input_stories'
os.makedirs(INPUT_DIR, exist_ok=True)

# --- Colab upload (skip if running locally) ---
try:
    from google.colab import files
    print('Upload your story JSONs:')
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.json'):
            with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
                f.write(uploaded[fname])
    print(f'Saved {len([f for f in uploaded if f.endswith(".json")])} JSONs to {INPUT_DIR}/')
except ImportError:
    print(f'Local mode - put your JSONs in ./{INPUT_DIR}/')

found = glob.glob(os.path.join(INPUT_DIR, '*.json'))
print(f'\n{len(found)} JSON files in {INPUT_DIR}/:')
for f in sorted(found):
    print(f'  {os.path.basename(f)}')

## 5. Audio Generation Functions

In [ ]:
async def make_audio(text, out_path):
    """Generate one MP3 from text using edge-tts."""
    communicate = edge_tts.Communicate(text, VOICE, rate=RATE, pitch=PITCH)
    await communicate.save(out_path)

def is_v1(fname):
    """A file is a '_1' file if its name (before .json) ends with _1."""
    stem = fname[:-5] if fname.endswith('.json') else fname
    return stem.endswith('_1')

async def process_set(file_list, out_dir, label):
    """Generate audio for every panel (and intro/conclusion) of every story."""
    os.makedirs(os.path.join(out_dir, 'audio'), exist_ok=True)
    os.makedirs(os.path.join(out_dir, 'stories'), exist_ok=True)
    
    for fpath in sorted(file_list):
        fname = os.path.basename(fpath)
        with open(fpath, encoding='utf-8') as f:
            story = json.load(f)
        pid = fname[:-5]   # strip .json, keep _1 if present
        title = story.get('story_context', {}).get('story_title', pid)
        panels = story.get('panels', [])
        print(f'\n[{label}] {title}  ({len(panels)} panels)')
        
        # Intro and conclusion audio
        ctx = story.get('story_context') or {}
        for key in ['introduction', 'conclusion']:
            text = ctx.get(key, '').strip()
            if text:
                audio_name = f'{pid}_{key}.mp3'
                print(f'  {key}...', end=' ')
                try:
                    await make_audio(text, os.path.join(out_dir, 'audio', audio_name))
                    ctx[f'{key}_audio'] = audio_name
                    print('OK')
                except Exception as e:
                    print(f'FAILED ({str(e)[:50]})')
        
        # Per-panel narration audio
        for panel in panels:
            n = panel['panel_number']
            text = panel.get('narrative_text', '').strip()
            if not text:
                panel['audio_file'] = None
                continue
            audio_name = f'{pid}_panel_{n}.mp3'
            print(f'  Panel {n}...', end=' ')
            try:
                await make_audio(text, os.path.join(out_dir, 'audio', audio_name))
                panel['audio_file'] = audio_name
                print('OK')
            except Exception as e:
                panel['audio_file'] = None
                print(f'FAILED ({str(e)[:50]})')
        
        story['story_context'] = ctx
        with open(os.path.join(out_dir, 'stories', f'{pid}.json'), 'w', encoding='utf-8') as f:
            json.dump(story, f, ensure_ascii=False, indent=2)
    print(f'\n[{label}] complete -> {out_dir}/')

print('Functions ready!')

## 6. Run — Generate All Audio

In [ ]:
async def run_all():
    all_files = glob.glob(os.path.join(INPUT_DIR, '*.json'))
    if not all_files:
        print(f'No JSON files in {INPUT_DIR}/')
        return
    base_files = [f for f in all_files if not is_v1(os.path.basename(f))]
    v1_files   = [f for f in all_files if is_v1(os.path.basename(f))]
    print(f'BASE set ({len(base_files)}) -> {BASE_OUT}/')
    print(f'V1 set ({len(v1_files)}) -> {V1_OUT}/')
    if base_files:
        await process_set(base_files, BASE_OUT, 'BASE')
    if v1_files:
        await process_set(v1_files, V1_OUT, 'V1')
    print('\n\nAll audio generated!')

# In Jupyter/Colab you can await at top level:
await run_all()
# If 'await' gives a syntax error in your environment, use instead:
# asyncio.run(run_all())

## 7. Preview — Listen to a Sample

In [ ]:
from IPython.display import Audio, display

# Play the first few generated clips so you can hear the voice
sample_mp3s = sorted(glob.glob(f'{BASE_OUT}/audio/*.mp3'))[:3]
for mp3 in sample_mp3s:
    print(os.path.basename(mp3))
    display(Audio(mp3))

## 8. (Colab) Download Both Folders

In [ ]:
import shutil
for folder in [BASE_OUT, V1_OUT]:
    if os.path.exists(folder):
        shutil.make_archive(folder, 'zip', folder)
        print(f'Created {folder}.zip')

try:
    from google.colab import files
    for folder in [BASE_OUT, V1_OUT]:
        if os.path.exists(f'{folder}.zip'):
            files.download(f'{folder}.zip')
except ImportError:
    print('Local mode - zips are in the working folder.')

print('\nEach has audio/ (MP3s) and stories/ (JSONs with audio_file fields).')